# Cross-camera GAP LeJEPA for cotton boll detection

This notebook **only configures and calls** functions from the `lejepa_cotton` package; no functions are defined here.

1. **Pretraining** - an empty-weight `yolov8n.yaml` backbone is trained with LeJEPA on synchronised frames from cameras 1, 2 and 4, using globally average-pooled P3/P4/P5 features and a cross-camera prediction loss plus SIGReg.
2. **Evaluation** - the LeJEPA backbone is compared with COCO `yolov8n.pt` weights on (a) cotton-boll detection fine-tuning and (b) a frozen-backbone linear probe for plot status.
3. **Visualization** - loss curves, camera-coloured PCA, detection curves/overlays, confusion matrices and metric bars.

Configuration cells and run cells are separate, so the evaluations can be run in a new session without re-running pretraining.

## 0. Install the package
Re-run after every push to the branch, then **restart the kernel**. `--force-reinstall --no-deps` fetches the latest commit even though the version number is unchanged (drop `--no-deps` on the very first install).

In [ ]:
BRANCH = "Packaging"
%pip install --force-reinstall --no-deps "git+https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git@{BRANCH}#subdirectory=lejepa_cotton_gap"

In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

import lejepa_cotton
from lejepa_cotton import (
    DetectionEvalConfig,
    PlotStatusDataset,
    PretrainConfig,
    ProbeEvalConfig,
    WeightSources,
    find_labeled_pairs,
    run_detection_evaluation,
    run_pretraining,
    run_probe_evaluation,
)

print("lejepa_cotton", lejepa_cotton.__version__, "from", lejepa_cotton.__file__)

## 1. Paths
All paths are absolute because the kernel does not start inside the repository. `os.chdir(OUTPUT_ROOT)` keeps files that Ultralytics writes to the working directory (e.g. the downloaded `yolov8n.pt`) out of the read-only `/`.

* **Detection dataset** - a folder of sub-datasets (`014/`, `2016-08-26/`, `ssl_active_1/`, ...), each with `images/` and `labels/`. Stray list files (`014.txt`) and the `.yaml` in the root are ignored; images duplicated across sub-datasets are kept once.
* **Plot status dataset** - a folder with the images and the annotation files; the JSON file is found automatically (`annotations.json` if present).

In [ ]:
HOME = Path.home()

MULTI_CAMERA_ROOT = HOME / "Downloads" / "LeJEPA _pretrainining_multi_camera_boll" / "mars_multi_camera_boll"
DETECTION_DATA = HOME / "Downloads" / "detection_dataset"
PLOT_STATUS_DIR = HOME / "Downloads" / "Plot Status"          # <- edit if different
OUTPUT_ROOT = HOME / "lejepa_cotton_outputs"                  # writable, outside the datasets

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_ROOT)

print("working directory:", Path.cwd())
print("multi-camera data found:", MULTI_CAMERA_ROOT.exists())
print("detection data found:   ", DETECTION_DATA.exists(), "| sub-datasets:", sorted(p.name for p in DETECTION_DATA.iterdir() if p.is_dir()) if DETECTION_DATA.exists() else "-")
print("plot-status data found: ", PLOT_STATUS_DIR.exists(), "| JSON files:", sorted(p.name for p in PLOT_STATUS_DIR.rglob("*.json")) if PLOT_STATUS_DIR.exists() else "-")

## 2. Configurations

In [ ]:
pretrain_cfg = PretrainConfig(
    image_root=MULTI_CAMERA_ROOT,
    output_dir=OUTPUT_ROOT / "pretraining",
    cameras=(1, 2, 4),        # 2nd, 3rd and 5th physical cameras (0-indexed in file names)
    max_samples=16_667,       # synchronised triplets -> ~50k images
    views_per_camera=1,
    image_size=128,
    batch_size=16,
    epochs=60,
    proj_dim=128,
    lr=1e-3,
    weight_decay=1e-4,
    lam=0.2,
    model_cfg="yolov8n.yaml",  # empty weights
)

# The LeJEPA checkpoint is referenced by path, so evaluation works in a fresh session.
weights = WeightSources(
    lejepa_checkpoint=pretrain_cfg.checkpoint_path,
    model_cfg="yolov8n.yaml",
    coco_weights="yolov8n.pt",
)

detection_cfg = DetectionEvalConfig(
    source_dir=DETECTION_DATA,
    output_dir=OUTPUT_ROOT / "detection",
    weights=weights,
    class_names=("cotton_boll",),
    variants=("lejepa", "coco_backbone"),   # neck + head random in both; add "coco" for the full COCO reference
    split_by="image",                # "folder" keeps each sub-dataset wholly in train or val
    val_ratio=0.2,                   # fixed validation split (never reduced)
    max_train_images=200,            # low-label regime; None = all training images
    image_size=256,
    batch_size=16,
    epochs=40,
    lr0=0.002,
    device="cpu",                    # failsafe; use "mps" if stable on your machine
)

probe_cfg = ProbeEvalConfig(
    image_dir=PLOT_STATUS_DIR,
    output_dir=OUTPUT_ROOT / "plot_status_probe",
    weights=weights,
    annotation_path=None,            # None = find the JSON inside PLOT_STATUS_DIR
    label_mapping={"headland": 0, "between_plots": 1, "in_plot": 2},
    variants=("lejepa", "coco"),
    scales=("P3", "P4", "P5"),
    image_size=256,
    epochs=50,
    test_ratio=0.5,                  # fixed test split (never reduced)
    max_train_images=200,            # stratified; None = all training images
)

## 2b. Check the evaluation datasets
Runs in seconds and fails early with a clear message if a dataset is not found or not readable, before any training starts.

In [ ]:
detection_pairs = find_labeled_pairs(DETECTION_DATA)
print("example pair:", detection_pairs[0][0].relative_to(DETECTION_DATA), "->", detection_pairs[0][1].relative_to(DETECTION_DATA))

plot_status_data = PlotStatusDataset(PLOT_STATUS_DIR, probe_cfg.annotation_path, probe_cfg.label_mapping, transform=None)


## 3. Cross-camera GAP LeJEPA pretraining
Skip this cell if the checkpoint already exists and you only want to re-run the evaluations. PCA snapshots are coloured by camera: well-mixed colours mean viewpoint-invariant embeddings.

In [ ]:
checkpoint, history = run_pretraining(pretrain_cfg)
history.tail()

In [ ]:
print("LeJEPA checkpoint:", weights.lejepa_checkpoint, "| exists:", weights.lejepa_checkpoint.exists())
display(Image(filename=str(pretrain_cfg.output_dir / "plots" / "loss_curves.png")))
print("Interactive PCA snapshots:", *sorted((pretrain_cfg.output_dir / "plots" / "pca_3d").glob("*.html")), sep="\n")

## 4. Evaluation A - cotton boll detection fine-tuning (LeJEPA vs COCO)
Both variants are fine-tuned on the **same** train/val split with identical hyper-parameters. By default the comparison is **`lejepa` vs `coco_backbone`**: only the backbone (layers 0-9) differs, and the neck and head start from random weights in both, so any difference comes from the backbone. Add `"coco"` (backbone, neck and head all COCO) as an upper reference, or `"scratch"` as a lower bound.

At the start of every run the model Ultralytics actually trains is compared with COCO, and the share of backbone/neck/head tensors identical to COCO is printed and stored in the summary (`coco_share_*`). Training stops with an error if a variant's neck or head is not random when it should be.

`max_train_images` caps only the training images. The validation split is fixed, so runs with different training-set sizes (e.g. 50, 100, 200, `None`) are scored on the same images and form a label-efficiency curve.

In [ ]:
detection_summary = run_detection_evaluation(detection_cfg)
detection_summary

In [ ]:
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_metrics.png")))
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_curves.png")))
print("Box overlays (green = ground truth, red = prediction):", detection_cfg.output_dir / "overlays")

## 5. Evaluation B - plot-status linear probe on frozen GAP features (LeJEPA vs COCO)
The backbone is frozen (including BatchNorm statistics); only a linear layer is trained on the concatenated P3/P4/P5 GAP vectors.

In [ ]:
probe_summary = run_probe_evaluation(probe_cfg)
probe_summary

In [ ]:
probe_plots = probe_cfg.output_dir / "plots"
display(Image(filename=str(probe_plots / "probe_metrics.png")))
display(Image(filename=str(probe_plots / "probe_loss_curves.png")))
for variant in probe_cfg.variants:
    display(Image(filename=str(probe_plots / f"confusion_{variant}.png")))
    display(Image(filename=str(probe_plots / f"pca2d_{variant}.png")))